# SolarSentinel – Getting Started

Welcome Team!

## What is This About?

We're building an **AI model that predicts solar storms faster** than currently possible.

- **The Problem:** Current forecasts only update every **3 hours**.  

- **Our Goal:** Forecasts every **few minutes** with real-time data.

- **The Tool:** The **Kp-Index** (0–9) measures how disturbed Earth's magnetic field is.

- **The Data:** 60 years of satellite data + real-time measurements

## Your Role

- **Alex:** : Github Master, code-master
- **Karl:** : Project-documentation  
- **Nooshin:** : Science behind it

*(Adjust as needed)*

## Setup: First Steps

In [2]:
# 2. Import essential libraries
import pandas as pd
import numpy as np
from pathlib import Path

print('✅ Libraries ready')

✅ Libraries ready


In [ ]:
# 3. Live data helper (NOAA SWPC + GFZ Potsdam, no API key needed)
import requests
import matplotlib.pyplot as plt

NOAA = "https://services.swpc.noaa.gov"

def get_json(url, **params):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

In [ ]:
# 4. NOAA storm classes (G-scale) by Kp
G_SCALE = {"G0 no storm": (0, 5), "G1 minor": (5, 6), "G2 moderate": (6, 7),
           "G3 strong": (7, 8), "G4 severe": (8, 9), "G5 extreme": (9, 10)}

def g_class(kp):
    # shortcut: Kp 5- (4.67) counts as G0 here, check NOAA's rule before using it in the model
    for name, (lo, hi) in G_SCALE.items():
        if lo <= kp < hi:
            return name

pd.DataFrame([{"class": n, "Kp from": lo, "Kp below": hi} for n, (lo, hi) in G_SCALE.items()])

In [ ]:
# 5. Official Kp from NOAA, last 7 days (time = start of each 3-hour window)
kp3 = pd.DataFrame(get_json(f"{NOAA}/products/noaa-planetary-k-index.json"))
kp3["time"] = pd.to_datetime(kp3["time_tag"], utc=True)
kp3["class"] = kp3["Kp"].apply(g_class)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(kp3["time"], kp3["Kp"], width=pd.Timedelta(hours=2.8), align="edge",
       color=["tab:red" if kp >= 5 else "tab:green" for kp in kp3["Kp"]])
ax.axhline(5, color="red", ls="--", label="storm threshold (G1)")
ax.set(ylim=(0, 9), ylabel="Kp", title="Official Kp: one value per 3 hours")
ax.legend()
plt.show()

In [ ]:
# 6. Today (UTC): how often was Kp in each storm class
today = pd.Timestamp.now(tz="UTC").normalize()
kp_today = kp3[kp3["time"] >= today]
print(f"{len(kp_today)} of 8 three-hour windows of today are published")
kp_today["class"].value_counts().reindex(list(G_SCALE), fill_value=0)

In [ ]:
# 7. NOAA estimated Kp every minute vs the official 3-hour steps
kp1 = pd.DataFrame(get_json(f"{NOAA}/json/planetary_k_index_1m.json"))
kp1["time"] = pd.to_datetime(kp1["time_tag"], utc=True)
kp_recent = kp3[kp3["time"] >= kp1["time"].min() - pd.Timedelta(hours=3)]
# repeat the last value 3 h later so the last window is drawn as a full step
kp_recent = pd.concat([kp_recent, kp_recent.tail(1).assign(time=lambda d: d["time"] + pd.Timedelta(hours=3))])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(kp1["time"], kp1["estimated_kp"], lw=0.8, label="estimated Kp (every minute)")
ax.step(kp_recent["time"], kp_recent["Kp"], where="post", lw=2, label="official Kp (every 3 h)")
ax.set(ylim=(0, 9), ylabel="Kp", title="Same index, two time resolutions")
ax.legend()
plt.show()

In [ ]:
# 8. GFZ Hp30: same idea as Kp, but every 30 minutes and open-ended above 9
t0 = today - pd.Timedelta(days=3)
hp = get_json("https://kp.gfz.de/app/json/", start=t0.strftime("%Y-%m-%dT%H:%M:%SZ"),
              end=pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%dT%H:%M:%SZ"), index="Hp30")
hp30 = pd.DataFrame({"time": pd.to_datetime(hp["datetime"], utc=True), "Hp30": hp["Hp30"]})
kp_3d = kp3[kp3["time"] >= t0]
kp_3d = pd.concat([kp_3d, kp_3d.tail(1).assign(time=lambda d: d["time"] + pd.Timedelta(hours=3))])

fig, ax = plt.subplots(figsize=(12, 4))
ax.step(hp30["time"], hp30["Hp30"], where="post", lw=1, label="Hp30 (every 30 min)")
ax.step(kp_3d["time"], kp_3d["Kp"], where="post", lw=2, label="Kp (every 3 h)")
ax.set(ylabel="index", title="Last 3 days: Hp30 shows what the 3-hour Kp smooths away")
ax.legend()
plt.show()

In [ ]:
# 9. Solar wind at L1 (~1.5 million km before Earth) and the resulting warning time
mag = pd.DataFrame(get_json(f"{NOAA}/json/rtsw/rtsw_mag_1m.json"))
wind = pd.DataFrame(get_json(f"{NOAA}/json/rtsw/rtsw_wind_1m.json"))
mag = mag[mag["active"]].copy()
wind = wind[wind["active"]].copy()
for df in (mag, wind):
    df["time"] = pd.to_datetime(df["time_tag"], utc=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(mag["time"], mag["bz_gsm"], lw=0.8)
ax1.axhline(0, color="grey")
ax1.set(ylabel="Bz GSM [nT]", title="Bz below 0 opens Earth's magnetic field to the solar wind")
ax2.plot(wind["time"], wind["proton_speed"], lw=0.8, color="tab:orange")
ax2.set(ylabel="speed [km/s]")
plt.show()

speed = wind.sort_values("time").tail(60)["proton_speed"].median()
print(f"Solar wind now ≈ {speed:.0f} km/s → arrives at Earth in ≈ {1.5e6 / speed / 60:.0f} min")

## Resources & Links

- **Project PDF:** `Solar Sentinel Live Geomagnetic Storm Forecasting.pdf`
- **Overview Notebook:** `SolarSentinel.ipynb`
- **Data:** Provided by Alex
- **GitHub:** See Issues & PRs on the project board

## Your Checklist for This Week

- [ ] Read this notebook
- [ ] Install dependencies
- [ ] Review `SolarSentinel.ipynb`
- [ ] Sync with your team
- [ ] Start your first task on the project board

## Questions?

Open an **Issue** on GitHub or chat with your team at the next meeting. 🚀